## 1. Dependencies Installation

In [1]:
%%capture
!pip install unsloth 

In [2]:
!pip install trl transformers polars pandas datasets tqdm --q

## 2. Initialize Model and Training Parameters

In [3]:
configurations = {
    "model_name": "unsloth/llama-3.2-3b-unsloth-bnb-4bit",
    "dtype": None,
    "max_seq_length": 512,
    "load_in_4bit": True,
    "learning_rate": 1e-4,
    "optimizer": "adamw_8bit",
    "batch_size": 2,
    "gradient_accumulation": 8,
    "epochs": 1,
    "decay": 0.01,
    "seed": 3407,
    "scheduler": "linear"
}

lora_config = {
    "lora_r": 16,
    "lora_alpha": 32,
    "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj", "up_proj", "down_proj", "gate_proj"],
    "lora_dropout": 0.05,
    "bias": "none",
    "use_rslora": False,
    "loftq_config": None
}

In [4]:
from unsloth import FastLanguageModel
from trl import SFTTrainer

model, tokenizer = FastLanguageModel.from_pretrained(
    configurations["model_name"],
    max_seq_length=configurations["max_seq_length"],
    load_in_4bit=configurations["load_in_4bit"],
    dtype=configurations["dtype"]
)

model = FastLanguageModel.get_peft_model(
    model,
    r=lora_config["lora_r"],
    target_modules=lora_config["target_modules"],
    lora_alpha=lora_config["lora_alpha"],
    lora_dropout=lora_config["lora_dropout"],
    bias=lora_config["bias"],
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=lora_config["use_rslora"],
    loftq_config=lora_config["loftq_config"]
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.8: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.35G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/230 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/459 [00:00<?, ?B/s]

Unsloth: Will load unsloth/llama-3.2-3b-unsloth-bnb-4bit as a legacy tokenizer.
Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.4.8 patched 28 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


## 3. Data Preparation

In [5]:
from datasets import Dataset, ClassLabel, load_dataset
import json


def get_category():
    map_datapath = "/kaggle/input/datasets/akuseru59/banking77-intent-pred/map.json"
    with open(map_datapath, 'r', encoding="utf-8") as inf:
        category_map = json.load(inf)

        return dict(category_map)

category_map = get_category()

train_datapath = "/kaggle/input/datasets/akuseru59/banking77-intent-pred/train.csv"
test_datapath = "/kaggle/input/datasets/akuseru59/banking77-intent-pred/test.csv"

def get_dataset(path):
    try:
        dataset = load_dataset("csv", data_files=path)["train"]

        def add_label(example):
            key = example["category"].strip().lower()
            example["label"] = category_map[key]
            return example

        dataset = dataset.map(add_label)

        return dataset

    except FileNotFoundError:
        raise ValueError(f"No file found at {path}")

train_dataset = get_dataset(train_datapath)
test = get_dataset(test_datapath)

num_classes = len(set(train_dataset["label"]))

train_dataset = train_dataset.cast_column(
    "label",
    ClassLabel(num_classes=num_classes)
)

split = train_dataset.train_test_split(
    test_size=0.1,
    seed=42,
    stratify_by_column="label"
)

validation = split["test"]
train = split["train"]

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/10003 [00:00<?, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/3080 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/10003 [00:00<?, ? examples/s]

In [6]:
PROMPT_TEMPLATE = """### Instruction:
Classify the intent of the following banking request.

### Input:
{}

### Response:
Answer: <|label|> {}"""

EOS_TOKEN = tokenizer.eos_token

def format_prompts(examples):
    texts = []
    for input_text, output_text in zip(examples["text"], examples["label"]):
        text = PROMPT_TEMPLATE.format(
            input_text,
            str(output_text),
        ) + EOS_TOKEN

        texts.append(text)

    return {"texts": texts}

validation = validation.map(format_prompts, batched=True)
train = train.map(format_prompts, batched=True)

Map:   0%|          | 0/1001 [00:00<?, ? examples/s]

Map:   0%|          | 0/9002 [00:00<?, ? examples/s]

In [7]:
def tokenize(example):
    text = example["texts"]

    marker = "<|label|>"
    start_char = text.find(marker)
    if start_char == -1:
        raise ValueError(f"Marker not found in: {text}")

    tokenized = tokenizer(
        text,
        truncation=True,
        max_length=256,
        return_offsets_mapping=True,
    )

    labels = tokenized["input_ids"].copy()
    offsets = tokenized["offset_mapping"]

    start_token = None
    for i, (s, e) in enumerate(offsets):
        if s <= start_char < e:
            start_token = i + 1 
            break

    if start_token is None:
        raise ValueError("Token alignment failed")

    for i in range(start_token):
        labels[i] = -100

    tokenized["labels"] = labels
    tokenized.pop("offset_mapping")

    return tokenized

train = train.map(tokenize)
validation = validation.map(tokenize)

Map:   0%|          | 0/9002 [00:00<?, ? examples/s]

Map:   0%|          | 0/1001 [00:00<?, ? examples/s]

In [8]:
from trl import SFTTrainer, SFTConfig
from unsloth import is_bf16_supported

training_args = SFTConfig(
    dataset_num_proc=2,
    packing=False,

    per_device_train_batch_size=configurations['batch_size'],
    gradient_accumulation_steps=configurations['gradient_accumulation'],

    warmup_steps=10,
    num_train_epochs=configurations["epochs"],
    learning_rate=configurations['learning_rate'],

    fp16=not is_bf16_supported(),
    bf16=is_bf16_supported(),

    logging_steps = 10,
    eval_strategy = "steps",
    eval_steps = 100,
    save_strategy = "steps",
    save_steps = 100,
    load_best_model_at_end = True,

    optim=configurations['optimizer'],
    weight_decay=configurations['decay'],
    lr_scheduler_type=configurations['scheduler'],

    seed=configurations["seed"],
    output_dir="/kaggle/working/"
)

FastLanguageModel.for_training(model)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train,
    eval_dataset=validation,
    args=training_args,
)

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


## 4. Train

In [ ]:
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 9,002 | Num Epochs = 1 | Total steps = 563
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss,Validation Loss
100,0.711194,0.580427


Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/checkpoint-100/tokenizer_config.json.


In [ ]:
import os

output_dir = "/kaggle/working/best-model"
os.makedirs(output_dir, exist_ok=True)

model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)